> **Reused upstream sample; local workshop adaptation.** Source: [Azure-Samples/azure-ai-content-understanding-python](https://github.com/Azure-Samples/azure-ai-content-understanding-python/blob/19f1a9d68ef5de66abef056564f32e4bc49f4067/notebooks/content_extraction.ipynb).
> Original commit: `19f1a9d68ef5de66abef056564f32e4bc49f4067`. MIT notice: [LICENSE](../LICENSE).
> Local changes: repo .venv/.env and Entra-only auth; read-only defaults; existing configured completion model; compact display previews with complete JSON saved by upstream helpers; explicit opt-in sample cleanup. Analysis calls incur Azure usage. No new processing framework.


# Extract Content from Your File

This notebook demonstrates how to use the Content Understanding API to extract semantic content from multimodal files.

## Prerequisites
1. Ensure your Azure AI service is configured by following the [configuration steps](../README.md#configure-azure-ai-service-resource).
2. Install the required packages to run this sample.

Use the repository's existing **.venv**. This upstream install cell is intentionally not executed; install only a specifically missing package in a visible terminal.


## Use the existing upstream REST client
These educational demos retain the upstream REST wrapper and analysis/schema code. This local setup reads the root .env, uses the shared .venv and Microsoft Entra ID only. The claims app continues to use the official SDK.


In [1]:
from datetime import datetime
from pathlib import Path
import logging
import json
import os
import sys
import time
import asyncio
from typing import Any, Optional
from dotenv import find_dotenv, load_dotenv

ENV_FILE = Path(find_dotenv(usecwd=True, raise_error_if_not_found=True))
load_dotenv(ENV_FILE, override=True)
NOTEBOOK_DIR = ENV_FILE.parent / "demo" / "content-understanding" / "notebooks"
os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(NOTEBOOK_DIR.parent / "python"))
from content_understanding_client import AzureContentUnderstandingClient
from extension.sample_helper import save_json_to_file
from azure.identity import DefaultAzureCredential

if Path(sys.executable).resolve() != (ENV_FILE.parent / ".venv" / "Scripts" / "python.exe").resolve():
    raise RuntimeError("Select the repository .venv kernel.")
logging.basicConfig(level=logging.WARNING)
AZURE_AI_ENDPOINT = os.environ["APP_CONTENT_UNDERSTANDING_ENDPOINT"]
API_VERSION = os.getenv("CU_API_VERSION", "2025-11-01")
credential = DefaultAzureCredential()
def token_provider():
    return credential.get_token("https://cognitiveservices.azure.com/.default").token

client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT, api_version=API_VERSION,
    token_provider=token_provider,
    x_ms_useragent="azure-ai-content-understanding-python-sample-ga",
)
RUN_SAMPLE_CLEANUP = False
print("Official upstream REST sample |", API_VERSION, "| Microsoft Entra ID")
print("Shared repo environment:", ENV_FILE.name, "| Kernel:", Path(sys.executable).relative_to(ENV_FILE.parent))


Official upstream REST sample | 2025-11-01 | Microsoft Entra ID
Shared repo environment: .env | Kernel: .venv\Scripts\python.exe


## Inspect existing model configuration (read-only)
The upstream default-update example is replaced here with a read-only check. No models are deployed and no shared defaults are changed. Custom examples use a model already mapped to the configured completion deployment.


In [2]:
defaults = client.get_defaults()
model_deployments = defaults.get("modelDeployments", {})
print("Existing resource model mappings (read-only):", model_deployments)
completion_deployment = os.environ["APP_CU_COMPLETION_DEPLOYMENT"]
completion_models = [model for model, deployment in model_deployments.items() if deployment == completion_deployment]
if not completion_models:
    raise ValueError("The configured completion deployment is not mapped in this account. Review resource defaults; this notebook will not patch them.")
COMPLETION_MODEL = completion_deployment if completion_deployment in completion_models else completion_models[0]
print("Custom sample completion model:", COMPLETION_MODEL, "| Existing deployment:", completion_deployment)


Existing resource model mappings (read-only): {'gpt-5.2': 'gpt-5.2', 'text-embedding-3-large': 'text-embedding-3-large', 'prebuilt-analyzer-completion': 'gpt-5.2', 'prebuilt-analyzer-completion-mini': 'gpt-5.2', 'prebuilt-analyzer-embedding': 'text-embedding-3-large'}
Custom sample completion model: gpt-5.2 | Existing deployment: gpt-5.2


## Document Content

The `prebuilt-documentSearch` analyzer transforms unstructured documents into structured, machine-readable data optimized for retrieval-augmented generation (RAG) and automated workflows. It extracts content and layout elements while preserving document structure and semantic relationships.

Key capabilities include:
1. **Content Analysis:** Extracts text (printed and handwritten), selection marks, barcodes (12+ types), mathematical formulas (LaTeX), hyperlinks, and annotations.
2. **Figure Analysis:** Generates descriptions for images/charts/diagrams, converts charts to Chart.js syntax, and diagrams to Mermaid.js syntax.
3. **Structure Analysis:** Identifies paragraphs with contextual roles (title, section heading, page header/footer), detects tables with complex layouts (merged cells, multi-page), and maps hierarchical sections.
4. **GitHub Flavored Markdown:** Outputs richly formatted markdown that preserves document structure for LLM comprehension and AI-powered analysis.
5. **Broad Format Support:** Processes PDFs, images, Office documents (Word, Excel, PowerPoint), text files (HTML, Markdown), structured files (XML, JSON, CSV), and email formats (EML, MSG).

For detailed information about document elements and markdown representation, see [Document elements](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/document/elements) and [Document markdown](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/document/markdown).

> **Note:** Figure analysis (descriptions and chart/diagram analysis) is only supported for PDF and image file formats.

In [3]:
# Analyze document from local file
analyzer_sample_file = '../data/invoice.pdf'
analyzer_id = 'prebuilt-documentSearch'

print(f"Analyzing {analyzer_sample_file} with {analyzer_id}...")
response = client.begin_analyze_binary(
    analyzer_id=analyzer_id,
    file_location=analyzer_sample_file,
)

result = client.poll_result(response)

print("\nMarkdown Content:")
print("=" * 50)
# Extract markdown from the first content element
contents = result.get("result", {}).get("contents", [])
if contents:
    content = contents[0]
    markdown = content.get("markdown", "")
    print(markdown[:1200])
print("=" * 50)

# Check if this is document content to access document-specific properties
if content.get("kind") == "document":
    document_content = content
    print(f"\nDocument Information:")
    print(f"Start page: {document_content.get('startPageNumber')}")
    print(f"End page: {document_content.get('endPageNumber')}")
    print(f"Total pages: {document_content.get('endPageNumber') - document_content.get('startPageNumber') + 1}")

    # Check for pages
    pages = document_content.get("pages")
    if pages is not None:
        print(f"\nPages ({len(pages)}):")
        for i, page in enumerate(pages):
            unit = document_content.get("unit", "units")
            print(f"  Page {page.get('pageNumber')}: {page.get('width')} x {page.get('height')} {unit}")

    # Check if there are tables in the document
    tables = document_content.get("tables")
    if tables is not None:
        print(f"\nTables ({len(tables)}):")
        table_counter = 1
        for table in tables:
            row_count = table.get("rowCount")
            col_count = table.get("columnCount")
            print(f"  Table {table_counter}: {row_count} rows x {col_count} columns")
            table_counter += 1
else:
    print("\nDocument Information: Not available for this content type")
    
# Save the result
saved_json_path = save_json_to_file(result, filename_prefix="content_analyzers_analyze_binary")
print(f"\nFull analysis result saved. Review the complete JSON at: {saved_json_path}")


Analyzing ../data/invoice.pdf with prebuilt-documentSearch...

Markdown Content:
CONTOSO LTD.


# INVOICE

Contoso Headquarters
123 456th St
New York, NY, 10001

INVOICE: INV-100

INVOICE DATE: 11/15/2019

DUE DATE: 12/15/2019

CUSTOMER NAME: MICROSOFT CORPORATION

SERVICE PERIOD: 10/14/2019 - 11/14/2019

CUSTOMER ID: CID-12345

Microsoft Corp
123 Other St,
Redmond WA, 98052

BILL TO:
Microsoft Finance
123 Bill St,
Redmond WA, 98052

SHIP TO:
Microsoft Delivery
123 Ship St,
Redmond WA, 98052

SERVICE ADDRESS:
Microsoft Services
123 Service St,
Redmond WA, 98052


<table>
<tr>
<th>SALESPERSON</th>
<th>P.O. NUMBER</th>
<th>REQUISITIONER</th>
<th>SHIPPED VIA</th>
<th>F.O.B. POINT</th>
<th>TERMS</th>
</tr>
<tr>
<td></td>
<td>PO-3333</td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
</table>


<table>
<tr>
<th>DATE</th>
<th>ITEM CODE</th>
<th>DESCRIPTION</th>
<th>QTY</th>
<th>UM</th>
<th>PRICE</th>
<th>TAX</th>
<th>AMOUNT</th>
</tr>
<tr>
<td>3/4/2021</td>
<td>A123</td>
<td>Consulting Servi

### Analyzing Documents from URLs

You can also analyze documents directly from publicly accessible URLs without downloading them first. This is useful for processing documents hosted on web servers, cloud storage, or GitHub repositories.

In [4]:
# Analyze document from URL
document_url = 'https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf'
analyzer_id = 'prebuilt-documentSearch'

print(f"Analyzing document from URL: {document_url}")
print(f"Using analyzer: {analyzer_id}\n")

response = client.begin_analyze_url(
    analyzer_id=analyzer_id,
    url=document_url,
)

result = client.poll_result(response)

print("\nMarkdown Content:")
print("=" * 50)
# Extract markdown from the first content element
contents = result.get("result", {}).get("contents", [])
if contents:
    content = contents[0]
    markdown = content.get("markdown", "")
    print(markdown[:1200])
print("=" * 50)

# Check if this is document content to access document-specific properties
if content.get("kind") == "document":
    document_content = content
    print(f"\nDocument Information:")
    print(f"Start page: {document_content.get('startPageNumber')}")
    print(f"End page: {document_content.get('endPageNumber')}")
    print(f"Total pages: {document_content.get('endPageNumber') - document_content.get('startPageNumber') + 1}")

    # Check for pages
    pages = document_content.get("pages")
    if pages is not None:
        print(f"\nPages ({len(pages)}):")
        for i, page in enumerate(pages):
            unit = document_content.get("unit", "units")
            print(f"  Page {page.get('pageNumber')}: {page.get('width')} x {page.get('height')} {unit}")

    # Check if there are tables in the document
    tables = document_content.get("tables")
    if tables is not None:
        print(f"\nTables ({len(tables)}):")
        table_counter = 1
        for table in tables:
            row_count = table.get("rowCount")
            col_count = table.get("columnCount")
            print(f"  Table {table_counter}: {row_count} rows x {col_count} columns")
            table_counter += 1
else:
    print("\nDocument Information: Not available for this content type")
    
# Save the result
saved_json_path = save_json_to_file(result, filename_prefix="content_analyzers_url_document")
print(f"\nFull analysis result saved. Review the complete JSON at: {saved_json_path}")


Analyzing document from URL: https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf
Using analyzer: prebuilt-documentSearch


Markdown Content:
CONTOSO LTD.


# INVOICE

Contoso Headquarters
123 456th St
New York, NY, 10001

INVOICE: INV-100

INVOICE DATE: 11/15/2019

DUE DATE: 12/15/2019

CUSTOMER NAME: MICROSOFT CORPORATION

SERVICE PERIOD: 10/14/2019 - 11/14/2019

CUSTOMER ID: CID-12345

Microsoft Corp
123 Other St,
Redmond WA, 98052

BILL TO:
Microsoft Finance
123 Bill St,
Redmond WA, 98052

SHIP TO:
Microsoft Delivery
123 Ship St,
Redmond WA, 98052

SERVICE ADDRESS:
Microsoft Services
123 Service St,
Redmond WA, 98052


<table>
<tr>
<th>SALESPERSON</th>
<th>P.O. NUMBER</th>
<th>REQUISITIONER</th>
<th>SHIPPED VIA</th>
<th>F.O.B. POINT</th>
<th>TERMS</th>
</tr>
<tr>
<td></td>
<td>PO-3333</td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
</table>


<table>
<tr>
<th>DATE</th>
<th>ITEM CODE</th>
<th>DESCRIPTION</th>
<th>QTY</th>
<th

## Audio Content
The `prebuilt-audioSearch` analyzer provides conversation analysis capabilities for audio files. It automatically transcribes audio content, performs speaker diarization to distinguish between speakers, and generates conversation summaries. The analyzer supports multilingual transcription and outputs transcripts in standard WebVTT format.

Key features include:
1. **Transcription:** Converts conversational audio into searchable text with sentence-level and word-level timestamps.
2. **Speaker Diarization:** Distinguishes between speakers in a conversation, attributing parts of the transcript to specific speakers (e.g., "Speaker 1", "Speaker 2").
3. **Timing Information:** Precise timing data in milliseconds (startTimeMs, endTimeMs) for each phrase, crucial for audio-text synchronization.
4. **Summary Generation:** Automatically generates a summary of the conversation for quick understanding.
5. **Multilingual Support:** Supports automatic language detection and multilingual transcription across multiple locales.
6. **Markdown Output:** Structured markdown format with WebVTT transcripts preserving speaker identification and timing.

For detailed information about audio markdown format and capabilities, see [Audio overview](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/audio/overview) and [AudioVisual markdown representation](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/video/markdown).

In [5]:
analyzer_sample_file = '../data/audio.wav'
analyzer_id = 'prebuilt-audioSearch'

# Analyze audio file with the created analyzer
print(f"Analyzing audio file from path: {analyzer_sample_file} with analyzer '{analyzer_id}'...")

# Begin audio analysis operation
print(f"Starting audio analysis with analyzer '{analyzer_id}'...")
analysis_response = client.begin_analyze_binary(
    analyzer_id=analyzer_id,
    file_location=analyzer_sample_file,
)

# Wait for analysis completion
print(f"Waiting for audio analysis to complete...")
analysis_result = client.poll_result(analysis_response)
print(f"Audio analysis completed successfully!")

print("\nMarkdown Content:")
print("=" * 50)
# Extract markdown from the first content element
contents = analysis_result.get("result", {}).get("contents", [])
if contents:
    content = contents[0]
    markdown = content.get("markdown", "")
    print(markdown[:1200])
print("=" * 50)

# Check if this is audio-visual content to access audio-visual properties
if content.get("kind") == "audioVisual":
    audio_visual_content = content
    print("\nAudio-Visual Information:")
    
    # Basic Audio-Visual Details
    try:
        start_time = audio_visual_content.get("startTimeMs")
        end_time = audio_visual_content.get("endTimeMs")
        duration_sec = (end_time - start_time) / 1000
        print(f"Start Time: {start_time} ms")
        print(f"End Time: {end_time} ms")
        print(f"Duration: {duration_sec:.2f} seconds")
    except (KeyError, TypeError):
        print("Missing basic audio-visual content details.")

    # Transcript Phrases (limit to 5)
    transcript_phrases = audio_visual_content.get("transcriptPhrases", [])
    if transcript_phrases:
        print(f"\nTranscript Phrases ({min(len(transcript_phrases), 5)}):")
        for idx, phrase in enumerate(transcript_phrases[:5]):
            print(f"  {idx + 1}. Speaker: {phrase.get('speaker')}")
            print(f"     Text: {phrase.get('text')}")
            print(f"     Start: {phrase.get('startTimeMs')} ms, End: {phrase.get('endTimeMs')} ms")
            confidence = phrase.get('confidence', 0)
            print(f"     Confidence: {confidence:.2%}")
            print(f"     Locale: {phrase.get('locale')}")
        if len(transcript_phrases) > 5:
            print(f"  ... and {len(transcript_phrases) - 5} more.")
    else:
        print("\nNo transcript phrases available.")

    # Markdown Preview
    if markdown:
        print("\nMarkdown Content Preview:")
        print(markdown[:200] + ("..." if len(markdown) > 200 else ""))
    else:
        print("\nNo Markdown content available.")
else:
    print("\nAudio-Visual Information: Not available for this content type.")

# Save the result
saved_json_path = save_json_to_file(analysis_result, filename_prefix="content_analyzers_audio")
print(f"\nFull analysis result saved. Review the complete JSON at: {saved_json_path}")



Analyzing audio file from path: ../data/audio.wav with analyzer 'prebuilt-audioSearch'...
Starting audio analysis with analyzer 'prebuilt-audioSearch'...
Waiting for audio analysis to complete...
Audio analysis completed successfully!

Markdown Content:
# Audio: 00:00.000 => 01:54.670

Transcript
```
WEBVTT

00:00.110 --> 00:02.190
<v Speaker 1>Thank you for calling Woodgrove Travel.

00:02.990 --> 00:04.590
<v Speaker 1>My name is Isabella Taylor.

00:05.310 --> 00:06.910
<v Speaker 1>How may I assist you today?

00:07.710 --> 00:10.270
<v Speaker 2>Hi Isabella, my name is John Smith.

00:11.070 --> 00:17.950
<v Speaker 2>I recently traveled from New York City to Los Angeles on a business trip, and I had a terrible experience with my flight.

00:18.750 --> 00:20.910
<v Speaker 1>I'm sorry to hear that, John.

00:21.710 --> 00:27.230
<v Speaker 1>Could you please provide me with the details of your flight, such as the airline name and flight number?

00:28.010 --> 00:31.370
<v Speaker 

## Video Content
The `prebuilt-videoSearch` analyzer provides comprehensive analysis of video content, combining visual frame extraction, audio transcription, and AI-powered insights. It transforms raw video into RAG-ready structured output in both Markdown and JSON formats, enabling applications like media asset management, content categorization, and retrieval-augmented generation.

Key features include:
1. **Transcription with Diarization:** Converts audio to searchable WebVTT transcripts with speaker identification and multilingual support (same language handling as audio).
2. **Key Frame Extraction:** Intelligently extracts representative frames (~1 FPS) from each scene, embedded as markdown image references with timestamps (e.g., `![](keyFrame.400.jpg)`).
3. **Shot Detection:** Identifies video segment boundaries aligned with camera cuts and scene transitions, providing timestamps in `cameraShotTimesMs`.
4. **Segment-based Analysis:** Analyzes multiple frames per segment to identify actions, events, and themes rather than individual frame analysis.
5. **Custom Field Extraction:** Define business-specific fields (brands, categories, sentiment) that the generative model extracts from visual and audio content.
6. **Structured Output:** Content organized in GitHub Flavored Markdown with precise temporal alignment and JSON with detailed metadata.

For detailed information about video capabilities, elements, and markdown format, see [Video overview](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/video/overview), [Video elements](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/video/elements), and [AudioVisual markdown representation](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/video/markdown).

In [6]:
from typing import Dict
from datetime import datetime


# Utility function to save keyframe images
def save_keyframe_image_to_file(
    image_content: bytes,
    keyframe_id: str,
    test_name: str,
    test_py_file_dir: str,
    identifier: Optional[str] = None,
    output_dir: str = "test_output",
) -> str:
    """Save keyframe image to output file using pytest naming convention.

    Args:
        image_content: The binary image content to save
        keyframe_id: The keyframe ID (e.g., "keyframes/733")
        test_name: Name of the test case (e.g., function name)
        test_py_file_dir: Directory where pytest files are located
        identifier: Optional unique identifier to avoid conflicts (e.g., analyzer_id)
        output_dir: Directory name to save the output file (default: "test_output")

    Returns:
        str: Path to the saved image file

    Raises:
        OSError: If there are issues creating directory or writing file
    """
    # Generate timestamp and frame ID
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # Extract the frame time from the keyframe path (e.g., "keyframes/733" -> "733")
    if "/" in keyframe_id:
        frame_id = keyframe_id.split("/")[-1]
    else:
        # Fallback: use as-is if no slash found
        frame_id = keyframe_id

    # Create output directory if it doesn't exist
    output_dir_path = os.path.join(test_py_file_dir, output_dir)
    os.makedirs(output_dir_path, exist_ok=True)

    # Generate output filename with optional identifier to avoid conflicts
    if identifier:
        output_filename = f"{test_name}_{identifier}_{timestamp}_{frame_id}.jpg"
    else:
        output_filename = f"{test_name}_{timestamp}_{frame_id}.jpg"

    saved_file_path = os.path.join(output_dir_path, output_filename)

    # Write the image content to file
    with open(saved_file_path, "wb") as image_file:
        image_file.write(image_content)

    print(f"Image file saved to: {saved_file_path}")
    return saved_file_path


analyzer_sample_file = '../data/FlightSimulator.mp4'
analyzer_id = 'prebuilt-videoSearch'

# Analyze video file with the created analyzer
print(f"Analyzing video file from path: {analyzer_sample_file} with analyzer '{analyzer_id}'...")

# Begin video analysis operation
print(f"Starting video analysis with analyzer '{analyzer_id}'...")
analysis_response = client.begin_analyze_binary(
    analyzer_id=analyzer_id,
    file_location=analyzer_sample_file,
)

# Wait for analysis completion
print(f"Waiting for video analysis to complete...")
analysis_result = client.poll_result(analysis_response)
print(f"Video analysis completed successfully!")

print("\nMarkdown Content:")
print("=" * 50)
# Extract markdown from the first content element
contents = analysis_result.get("result", {}).get("contents", [])
if contents:
    content = contents[0]
    markdown = content.get("markdown", "")
    print(markdown[:1200])
print("=" * 50)

# Check if this is video-visual content to access video-visual properties
if content.get("kind") == "audioVisual":
    video_visual_content = content
    print("\nVideo-Visual Information:")

    # Basic Video-Visual Details
    try:
        start_time = video_visual_content.get("startTimeMs")
        end_time = video_visual_content.get("endTimeMs")
        duration_sec = (end_time - start_time) / 1000
        print(f"Start Time: {start_time} ms")
        print(f"End Time: {end_time} ms")
        print(f"Duration: {duration_sec:.2f} seconds")
    except (KeyError, TypeError):
        print("Missing basic audio-visual content details.")

    # Transcript Phrases (limit to 5)
    transcript_phrases = video_visual_content.get("transcriptPhrases", [])
    if transcript_phrases:
        print(f"\nTranscript Phrases ({min(len(transcript_phrases), 5)}):")
        for idx, phrase in enumerate(transcript_phrases[:5]):
            print(f"  {idx + 1}. Speaker: {phrase.get('speaker')}")
            print(f"     Text: {phrase.get('text')}")
            print(f"     Start: {phrase.get('startTimeMs')} ms, End: {phrase.get('endTimeMs')} ms")
            confidence = phrase.get('confidence', 0)
            print(f"     Confidence: {confidence:.2%}")
            print(f"     Locale: {phrase.get('locale')}")
        if len(transcript_phrases) > 5:
            print(f"  ... and {len(transcript_phrases) - 5} more.")
    else:
        print("\nNo transcript phrases available.")

    # Key Frames (support both keyFrameTimesMs and KeyFrameTimesMs for forward compatibility)
    key_frame_times_ms = video_visual_content.get("keyFrameTimesMs") or video_visual_content.get("KeyFrameTimesMs", [])
    if key_frame_times_ms:
        print(f"\nKey Frames ({len(key_frame_times_ms)}):")
        for idx, key_frame_time in enumerate(key_frame_times_ms[:5]):
            print(f"  Frame {idx + 1}: Time {key_frame_time} ms")
        if len(key_frame_times_ms) > 5:
            print(f"  ... and {len(key_frame_times_ms) - 5} more.")
    else:
        print("\nNo key frame data available.")

    # Markdown Preview
    if markdown:
        print("\nMarkdown Content Preview:")
        print(markdown[:200] + ("..." if len(markdown) > 200 else ""))
    else:
        print("\nNo Markdown content available.")
else:
    print("\nVideo-Visual Information: Not available for this content type.")

# Save the result
saved_json_path = save_json_to_file(analysis_result, filename_prefix="content_analyzers_video")
print(f"\nFull analysis result saved. Review the complete JSON at: {saved_json_path}")

# Keyframe Processing
def extract_keyframe_ids(analysis_result: Dict[str, Any]) -> list[str]:
    """
    Extract all keyframe IDs from the analysis result.

    Args:
        analysis_result: The analysis result from the analyzer
    Returns:
        List of keyframe IDs (e.g., 'keyframes/1000', 'keyframes/2000')
    """
    print("Starting keyframe extraction from analysis result...")
    keyframe_ids = []
    contents = analysis_result.get("result", {}).get("contents", [])
    for idx, content in enumerate(contents):
        if content.get("kind") == "audioVisual":
            print(f"Found audioVisual content at index {idx}:")
            # Support both keyFrameTimesMs and KeyFrameTimesMs for forward compatibility
            key_frame_times_ms = content.get("keyFrameTimesMs") or content.get("KeyFrameTimesMs", [])
            if key_frame_times_ms:
                print(f"  Found {len(key_frame_times_ms)} keyframe timestamps")
                for time_ms in key_frame_times_ms:
                    keyframe_id = f"keyframes/{time_ms}"
                    keyframe_ids.append(keyframe_id)
            else:
                print(f"  No keyframe timestamps found in this audioVisual content.")
    print(f"Extracted {len(keyframe_ids)} total keyframe IDs; first five: {keyframe_ids[:5]}")
    return keyframe_ids

keyframe_ids = extract_keyframe_ids(analysis_result)
if keyframe_ids:
    print(f"\nDownloading {len(keyframe_ids)} keyframe images...")

    files_to_download = keyframe_ids[:min(3, len(keyframe_ids))]
    print(f"Files to download (first {len(files_to_download)}): {files_to_download}")

    for keyframe_id in files_to_download:
        print(f"Getting result file: {keyframe_id}")

        # Get the result file (keyframe image)
        image_content = client.get_result_file(
            analyze_response=analysis_response,
            file_id=keyframe_id,
        )

        if image_content:
            print(f"Retrieved image file for {keyframe_id} ({len(image_content)} bytes)")

            # Save the image file
            saved_file_path = save_keyframe_image_to_file(
                image_content=image_content,
                keyframe_id=keyframe_id,
                test_name="content_extraction_video",
                test_py_file_dir=os.getcwd(),
                identifier=analyzer_id
            )
            print(f"Saved keyframe image to: {saved_file_path}")
        else:
            print(f"No image content retrieved for keyframe: {keyframe_id}")
else:
    print("\nNo keyframe IDs found in analysis result.")


Analyzing video file from path: ../data/FlightSimulator.mp4 with analyzer 'prebuilt-videoSearch'...
Starting video analysis with analyzer 'prebuilt-videoSearch'...
Waiting for video analysis to complete...
Video analysis completed successfully!

Markdown Content:
# Video: 00:00.733 => 00:02.067
Width: 1080
Height: 608

Transcript
```
WEBVTT

00:01.390 --> 00:06.670
<Speaker 1>When it comes to the neural TTS, in order to get a good voice, it's better to have good data.
```

Key Frames
- 00:00.733 ![](keyFrame.733.jpg)
- 00:02.067 ![](keyFrame.2067.jpg)

Video-Visual Information:
Start Time: 733 ms
End Time: 2067 ms
Duration: 1.33 seconds

Transcript Phrases (1):
  1. Speaker: Speaker 1
     Text: When it comes to the neural TTS, in order to get a good voice, it's better to have good data.
     Start: 1390 ms, End: 6670 ms
     Confidence: 92.60%
     Locale: en-US

Key Frames (2):
  Frame 1: Time 733 ms
  Frame 2: Time 2067 ms

Markdown Content Preview:
# Video: 00:00.733 => 00:02.067
W

## Congratulations!

You've successfully learned how to extract content from multimodal files using Azure Content Understanding! You explored:

- **Document extraction** with the `prebuilt-documentSearch` analyzer
- **Audio transcription** with speaker diarization using `prebuilt-audioSearch`
- **Video analysis** with keyframe extraction using `prebuilt-videoSearch`

### Learn More

To dive deeper into Azure Content Understanding capabilities:

- **[Content Understanding Overview](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/overview)** - Comprehensive introduction to the service
- **[What's New](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/whats-new)** - Latest features and updates
- **[Content Extraction Guide](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/document/overview#content-extraction)** - Detailed documentation on extraction capabilities

Explore other notebooks in this repository to learn about custom analyzers, field extraction, and advanced scenarios!